In [30]:
import pandas as pd
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression

In [31]:
#  Define file paths for dataset, model, and supporting files
DATA_FILE = "data_balanced.csv"
MODEL_FILE = "model.pkl"
COLUMNS_FILE = "columns.pkl"
ENCODERS_FILE = "encoders.pkl"

In [32]:
#  Load dataset and display its shape (rows, columns)
print("Loading data...")
df = pd.read_csv(DATA_FILE, low_memory=False)
print(f"Data shape: {df.shape}")

Loading data...
Data shape: (66836, 43)


In [33]:
#  Validate that target column 'loanstatus' exists in dataset
if 'loanstatus' not in df.columns:
    print("Error: 'loanstatus' column not found in data!")
    exit(1)

In [34]:
#  Create target variable 'credit_risk' (1 = default, 0 = non-default)
df['credit_risk'] = df['loanstatus'].apply(
    lambda x: 1 if x == 'CHGOFF' else 0
)
print(f"Class distribution:\n{df['credit_risk'].value_counts()}")

Class distribution:
credit_risk
0    33418
1    33418
Name: count, dtype: int64


In [35]:
#Feature selection

In [36]:
#  Select relevant features for training the model
features = [
    'grossapproval',
    'terminmonths',
    'initialinterestrate',
    'businesstype',
    'naicsdescription',
    'jobssupported',
    'collateralind',
    'revolverstatus'
]
missing_features = [f for f in features if f not in df.columns]
if missing_features:
    print(f"Error: Missing features: {missing_features}")
    exit(1)

df = df[features + ['credit_risk']]
print (df)

       grossapproval  terminmonths  initialinterestrate businesstype  \
0              10000            84                 9.49  CORPORATION   
1              30000            31                 6.50  CORPORATION   
2             164000           203                 6.50  CORPORATION   
3              55600            46                 9.95  CORPORATION   
4             275000           116                 5.50  CORPORATION   
...              ...           ...                  ...          ...   
66831         179000             1                 7.02  CORPORATION   
66832         490000           120                 7.75  CORPORATION   
66833         250000             2                 6.00  CORPORATION   
66834         245000            60                 5.90  CORPORATION   
66835         145800            60                 6.60  CORPORATION   

                                        naicsdescription  jobssupported  \
0                 Painting and Wall Covering Contractors    

In [37]:
#  Handle missing values
print("Handling missing values...")
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].fillna(df[col].median())
    else:
        df[col] = df[col].fillna("Unknown")

print(f"Missing values after handling:\n{df.isnull().sum()}")

Handling missing values...
Missing values after handling:
grossapproval          0
terminmonths           0
initialinterestrate    0
businesstype           0
naicsdescription       0
jobssupported          0
collateralind          0
revolverstatus         0
credit_risk            0
dtype: int64


In [38]:
#  Encoding
print("Encoding categorical features...")
encoders = {}
categorical_cols = ['businesstype', 'naicsdescription', 'collateralind']

for col in categorical_cols:
    if col in df.columns:
        encoder = LabelEncoder()
        df[col] = encoder.fit_transform(df[col].astype(str))
        encoders[col] = encoder


Encoding categorical features...


In [39]:
# Save encoders
pickle.dump(encoders, open(ENCODERS_FILE, "wb"))

#  Features & target
X = df.drop('credit_risk', axis=1)
y = df['credit_risk']

In [40]:
# Save columns
pickle.dump(X.columns.tolist(), open(COLUMNS_FILE, "wb"))

print(f"Features for training: {X.columns.tolist()}")

#  Train-test split
print("Splitting data...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

Features for training: ['grossapproval', 'terminmonths', 'initialinterestrate', 'businesstype', 'naicsdescription', 'jobssupported', 'collateralind', 'revolverstatus']
Splitting data...
Training set: (53468, 8), Test set: (13368, 8)


In [41]:
#  Logistic Regression (MAIN MODEL)
print("Training Logistic Regression model...")

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

print("✅ Model trained successfully!")


Training Logistic Regression model...
✅ Model trained successfully!


In [42]:
#  Evaluation
print("Evaluating model...")
y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.3).astype(int)  # Lower threshold

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Low Risk', 'High Risk']))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Evaluating model...
Accuracy: 0.6443

Classification Report:
              precision    recall  f1-score   support

    Low Risk       0.85      0.35      0.50      6684
   High Risk       0.59      0.94      0.73      6684

    accuracy                           0.64     13368
   macro avg       0.72      0.64      0.61     13368
weighted avg       0.72      0.64      0.61     13368


Confusion Matrix:
[[2336 4348]
 [ 407 6277]]


In [43]:
#  Save model
print(f"Saving model to '{MODEL_FILE}'...")
pickle.dump(model, open(MODEL_FILE, "wb"))

print("✅ All files saved successfully!")

Saving model to 'model.pkl'...
✅ All files saved successfully!
